# 2. 도구 설계

모델은 함수 대신 이름, 설명, 스키마와 오류 문자열을 본다.
도구 수, 이름과 설명, 인자 검증, 오류 메시지, 결과 크기, 승인 게이트를 차례로 측정한다.

### 공통 설정

`1`과 같은 블록이다. 노트북끼리 코드를 공유하지 않는다.

In [ ]:
import json
import os
import shutil
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
from pydantic import BaseModel, ValidationError, field_validator

load_dotenv(find_dotenv(".env", usecwd=True), override=True)
MODEL = os.environ["OPENAI_DEFAULT_MODEL"]

client = OpenAI()

PRICES = {MODEL: (2.50, 15.00)}  # 1M 토큰당 (입력, 출력) USD
USAGE = []


def keep(resp):
    USAGE.append((resp.model, resp.usage))
    return resp

코퍼스와 이 노트북의 과제다. 도구 셋은 `1`과 같다.

In [ ]:
CORPUS = Path("../../data/corpus/graph")
TASK = "한국독립당을 언급한 문서가 몇 편인지 세고 숫자로 답한다"


def grep_docs(pattern):
    return [{"title": f.stem, "count": f.read_text(encoding="utf-8").count(pattern)}
            for f in sorted(CORPUS.glob("*.md")) if pattern in f.read_text(encoding="utf-8")]


def list_docs():
    return [f.stem for f in sorted(CORPUS.glob("*.md"))]


def read_doc(title):
    return (CORPUS / f"{title}.md").read_text(encoding="utf-8")[:2000]

### 루프는 `1`과 같다

바뀌는 것은 도구 쪽뿐이다. 어느 도구를 골랐는지 순서대로 돌려주게만 고쳤다. 이 회차의 측정값이 그것이다.

In [ ]:
def run(task, tools, registry, max_turns=6):
    picked = []
    r = keep(client.responses.create(model=MODEL, input=task, tools=tools))
    for _ in range(max_turns):
        calls = [i for i in r.output if i.type == "function_call"]
        picked += [c.name for c in calls]
        if not calls:
            return r.output_text, picked
        out = [{"type": "function_call_output", "call_id": c.call_id,
                "output": registry(c.name, json.loads(c.arguments))} for c in calls]
        r = keep(client.responses.create(model=MODEL, previous_response_id=r.id,
                                         tools=tools, input=out))
    return None, picked

도구 정의를 여러 벌 만들어야 하므로 정의를 찍어내는 함수를 둔다.

In [ ]:
def spec(name, desc, props):
    return {"type": "function", "name": name, "description": desc, "strict": True,
            "parameters": {"type": "object", "properties": props,
                           "required": list(props), "additionalProperties": False}}


TEXT = {"pattern": {"type": "string", "description": "찾을 낱말"}}
TITLE = {"title": {"type": "string", "description": "문서 제목"}}
print(spec("grep_docs", "낱말이 들어 있는 문서의 제목과 등장 횟수를 돌려준다.", TEXT))

### 1. 도구가 늘면 고르는 일이 어려워진다

같은 과제를 도구 셋과 도구 아홉으로 준다. 늘린 여섯은 쓸모없는 것이 아니라
grep과 겹치는 것들이다. 실무에서 도구가 느는 방식이 그렇다.

In [ ]:
FEW = [spec("grep_docs", "낱말이 들어 있는 문서의 제목과 등장 횟수를 돌려준다.", TEXT),
       spec("list_docs", "코퍼스에 있는 문서 제목을 모두 돌려준다.", {}),
       spec("read_doc", "문서 한 편의 본문 앞부분을 돌려준다.", TITLE)]

MANY = FEW + [
    spec("search_docs", "질의어로 문서를 찾는다.", {"query": {"type": "string", "description": "질의어"}}),
    spec("find_text", "본문에서 문자열을 찾는다.", TEXT),
    spec("count_term", "용어의 빈도를 센다.", TEXT),
    spec("scan_corpus", "코퍼스 전체를 훑는다.", TEXT),
    spec("lookup_entity", "인물이나 단체를 조회한다.", {"name": {"type": "string", "description": "이름"}}),
    spec("summarize_doc", "문서를 요약한다.", TITLE),
]
print([t["name"] for t in MANY])

아홉 개 중 실제로 구현된 것은 셋뿐이다. 나머지를 부르면 오류 문자열이 돌아간다.

In [ ]:
BASE = {"grep_docs": grep_docs, "list_docs": list_docs, "read_doc": read_doc}


def plain(name, args):
    if name not in BASE:
        return "error: 없는 도구다."
    try:
        return json.dumps(BASE[name](**args), ensure_ascii=False)
    except Exception as e:
        return f"error: {type(e).__name__}"

도구 셋으로 한 번 돌린다. 고른 도구의 순서와 답을 본다.

In [ ]:
answer, picked = run(TASK, FEW, plain)
print("고른 도구", picked)
print("답", answer)

단일 실행의 차이는 실행 편차와 구분하기 어려우므로 두 벌을 세 번씩 돌린다. 정식 측정은 Part 4 `15`의 태스크셋에서 한다.

In [ ]:
for label, tools in [("도구 3개", FEW), ("도구 9개", MANY)]:
    for i in range(3):
        answer, picked = run(TASK, tools, plain)
        print(f"{label}  {i + 1}회  고른 도구 {picked}  답 {str(answer)[:24]}")

### 2. 이름과 설명이 곧 프롬프트다

같은 함수를 두 이름으로 등록하고 둘 다 준다. 어느 쪽을 불러도 실행되는 것은 같은 `grep_docs`다.
모델이 무엇을 고르는지가 이름과 설명이 하는 일의 전부다.

In [ ]:
MIXED = [spec("process", "입력을 처리한다.", TEXT),
         spec("grep_docs", "낱말이 들어 있는 문서의 제목과 등장 횟수를 돌려준다."
                           " 코퍼스 전체를 훑으므로 개수를 세는 질문에 쓴다.", TEXT)]


def twin(name, args):
    return json.dumps(grep_docs(args["pattern"]), ensure_ascii=False)

세 번 돌려 어느 이름을 고르는지 본다.

In [ ]:
for i in range(3):
    answer, picked = run(TASK, MIXED, twin)
    print(f"{i + 1}회  고른 도구 {picked}  답 {str(answer)[:20]}")

### 3. 인자는 도구 안에서 검증한다

`1`의 마지막에서 `read_doc`이 코퍼스 밖 파일을 읽는 것을 봤다.
모델이 나쁜 인자를 보내서가 아니라 검증하는 자리가 없어서였다.
스키마는 타입만 본다. 경로가 코퍼스 안인지는 스키마가 모른다.

In [ ]:
from pydantic import BaseModel, ValidationError, field_validator
from pydantic_core import PydanticCustomError


class ReadDoc(BaseModel):
    title: str

    @field_validator("title")
    @classmethod
    def inside_corpus(cls, v):
        root = CORPUS.resolve()
        target = (root / f"{v}.md").resolve()
        if target.parent != root:
            raise PydanticCustomError(
                "outside_corpus", "'{title}' 는 코퍼스 밖을 가리킨다", {"title": v})
        if not target.is_file():
            raise PydanticCustomError(
                "not_found", "'{title}' 문서가 없다", {"title": v})
        return v


RECOVERY = {
    "not_found": ("error: '{title}' 라는 제목의 문서가 없다."
                  " list_docs 로 제목을 확인하고 그대로 다시 부른다."),
    "outside_corpus": ("error: '{title}' 는 제목이 아니라 경로로 해석된다."
                       " '/' 나 '..' 없이 list_docs 에 나온 제목을 그대로 넣는다."),
    "missing": "error: title 인자가 없다. read_doc(title=...) 형식으로 다시 부른다.",
}
FALLBACK = "error: title 은 문자열이어야 한다. list_docs 로 제목을 확인한다."



정상 제목과 상위 경로를 넣어 본다.

In [ ]:
for arg in ["김구", "../README"]:
    try:
        print("통과", ReadDoc(title=arg).title)
    except ValidationError as e:
        print("거부", e.errors()[0]["msg"])

### 4. 오류 메시지가 곧 복구 지시문이다

없는 제목을 읽게 만들고 세 가지를 되돌린다. 도구 본문은 같고 실패했을 때의 문자열만 다르다.

In [ ]:
def make_registry(style):
    def registry(name, args):
        if name == "read_doc":
            try:
                ReadDoc(**args)
            except ValidationError:
                if style == "숨김":
                    return ""
                if style == "짧게":
                    return "error: invalid input"
                return (f"error: '{args.get('title')}' 라는 제목의 문서가 없다."
                        " list_docs 로 제목을 확인하고 그대로 다시 부른다.")
        return plain(name, args)
    return registry


print(make_registry("복구 지시")("read_doc", {"title": "김구 (독립운동가)"}))

세 형식으로 같은 과제를 돌리고 고른 도구의 순서를 비교한다.

In [ ]:
for style in ["숨김", "짧게", "복구 지시"]:
    answer, picked = run("'김구 (독립운동가)' 문서를 읽고 첫 문장을 그대로 옮긴다",
                         FEW, make_registry(style))
    print(f"{style:6} {picked}")
    print(f"       {str(answer)[:64]}")

좋은 오류 메시지가 복구율을 높이는지는 모델과 과제마다 측정해야 한다. 이 과제는 도구가 적고 복구 경로가 단순해 형식의 차이가 잘 드러나지 않는다.
오류 내용과 관계없이 예외는 문자열로 돌려줘야 한다. 예외가 루프를 중단하면 모델에 복구 기회가 없다.

### 5. 결과 크기도 예산이다

도구가 돌려주는 것이 그대로 다음 요청의 입력이 된다.
자를 때는 잘랐다는 사실을 결과에 적는다. 적지 않으면 모델이 그것을 전문으로 믿는다.

In [ ]:
def read_capped(title, cap=1500):
    text = (CORPUS / f"{title}.md").read_text(encoding="utf-8")
    if len(text) <= cap:
        return text
    return text[:cap] + f"\n... [잘림. 전체 {len(text):,}자 중 앞 {cap:,}자]"


raw = (CORPUS / "안중근.md").read_text(encoding="utf-8")
print(f"안중근.md {len(raw):,}자")
print(read_capped("안중근")[-60:])

전문과 잘라낸 본문으로 같은 요약을 시킨다. 입력 토큰 차이를 본다.

In [ ]:
for label, text in [("전문", raw), ("잘라서", read_capped("안중근"))]:
    r = keep(client.responses.create(model=MODEL, input=f"다음 글을 한 문장으로 요약한다.\n\n{text}"))
    print(f"{label:4} 입력 {r.usage.input_tokens:6,d} 토큰   {r.output_text[:44]}")

### 6. 부작용이 있는 도구는 게이트를 지난다

읽기는 되돌릴 수 있고 쓰기는 되돌릴 수 없다. 그 경계에 사람을 세운다. (human-in-the-loop)
게이트를 거부로 만들면 모델이 무엇을 하려 했는지가 오류 문자열로 남는다.

In [ ]:
WORK = Path("work")
WORK.mkdir(exist_ok=True)


def write_note(name, text):
    (WORK / name).write_text(text, encoding="utf-8")
    return f"{name} 에 {len(text)}자를 썼다"


TOOLS_W = FEW + [spec("write_note", "메모를 파일로 저장한다.",
                      {"name": {"type": "string", "description": "파일 이름"},
                       "text": {"type": "string", "description": "저장할 내용"}})]

게이트는 레지스트리 안에 있다. 승인이 없으면 실행하지 않고 요청만 기록한다.

In [ ]:
GATED = {"write_note"}
requested = []


def registry_gated(name, args, approve=False):
    if name in GATED:
        requested.append((name, args))
        if not approve:
            return ("error: 이 도구는 사람의 승인이 필요하다."
                    " 무엇을 어떤 이름으로 저장할지 사용자에게 먼저 확인한다.")
        return write_note(**args)
    return plain(name, args)

거부 상태로 돌린다. 모델이 무엇을 저장하려 했는지 `requested`에 남는다.

In [ ]:
answer, picked = run("한국독립당을 언급한 문서 수를 세어 count.txt 에 저장한다",
                     TOOLS_W, lambda n, a: registry_gated(n, a, approve=False))
print("고른 도구", picked)
print("막힌 요청", requested)
print("모델의 답", str(answer)[:120])

승인 상태로 돌린다. 파일이 실제로 생긴다.

In [ ]:
answer, picked = run("한국독립당을 언급한 문서 수를 세어 count.txt 에 저장한다",
                     TOOLS_W, lambda n, a: registry_gated(n, a, approve=True))
print("고른 도구", picked)
print("파일", [f.name for f in WORK.iterdir()], (WORK / "count.txt").read_text(encoding="utf-8"))

작업 디렉터리를 정리한다.

In [ ]:
shutil.rmtree(WORK)

### ToolRegistry 구성

| 항목 | 어디서 |
|---|---|
| 이름과 설명 | `spec`. 모델이 읽는 유일한 문서 |
| 인자 검증 | `ReadDoc`. 스키마가 못 보는 제약을 여기서 본다 |
| 오류 문자열 | `make_registry`. 복구 지시문으로 쓴다 |
| 결과 크기 제한 | `read_capped`. 잘랐다는 사실을 같이 돌려준다 |
| 권한 | `GATED`. 부작용이 있는 것만 승인을 받는다 |

이 다섯 항목이 `5`의 ToolRegistry 인터페이스가 된다. 여기서는 승인 여부를 인자로 전달하며 실제 사용자 승인 게이트는 Part 4 `14`에서 구현한다.

### 비용 확인

`keep`으로 모아 둔 usage에 단가를 곱해 합산한다.

In [ ]:
total = 0.0
for name, u in USAGE:
    p = next((PRICES[k] for k in sorted(PRICES, key=len, reverse=True) if name.startswith(k)), None)
    if p:
        total += u.input_tokens / 1e6 * p[0] + u.output_tokens / 1e6 * p[1]

print(f"호출 {len(USAGE)}회")
print(f"입력 {sum(u.input_tokens for _, u in USAGE):,} 토큰   출력 {sum(u.output_tokens for _, u in USAGE):,} 토큰")
print(f"누적 ${total:.4f}")